In [15]:
from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter
from langchain.agents import create_agent
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.structured_output import ToolStrategy
load_dotenv()

True

In [11]:
def get_weather(city: str) -> str:
    """" Get weather for a given city """
    return f"The weather in {city} is sunny"

model= ChatOpenRouter( model="arcee-ai/trinity-large-preview:free")

agent= create_agent(model=model, tools=[get_weather], system_prompt="You are a helpful agent")
agent.invoke({"messages": [{"role": "user", "content": "what is the weather in sf"}]})

{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='ffe46a7e-d7f1-4863-a993-4ad7c6b4a5c5'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'arcee-ai/trinity-large-preview:free', 'id': 'gen-1772101514-RTfeGLz2veOchMLzwP0O', 'created': 1772101514, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter'}, id='lc_run--019c997b-1f58-7130-a93b-cf96eeca25b3-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'call-8e12f957-85db-4078-a204-e7115eba0632', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 141, 'output_tokens': 21, 'total_tokens': 162, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 0}}),
  ToolMessage(content='The weather in San Francisco is sunny', name='get_weather', id='273658e1-b6cb-450e-a16b-7c08bc9d5e45', tool_call_id='call-8e12f957-85db-4078-a

In [60]:
SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. 
If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."""

@tool
def get_weather_for_location(location: str) -> str:
    """" Get weather for a given location. Assume exact location if necessary. """
    return f"The weather in {location} is sunny"

@dataclass
class Context:
    """" Custom runtime context schema. """
    user_id: str

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """" Get the user's location based on user ID. """
    user_id= runtime.context.user_id
    return "Florida" if user_id=="1" else "New York"

model= ChatOpenRouter( model="stepfun/step-3.5-flash:free", temperature= 1, max_tokens=1000)

@dataclass
class ResponseFormat:
    """" Response schema for the agent. """
    # It always need to be punny (Required)
    punny_response: str
    #Any interesting information about the weather if available
    weather_conditions: str|None= None

checkpointer= InMemorySaver()

In [61]:
agent= create_agent(
    model= model, 
    system_prompt= SYSTEM_PROMPT, 
    tools=[get_user_location, get_weather_for_location], 
    context_schema= Context, 
    response_format= ResponseFormat,
    checkpointer= checkpointer
    )

config= {"configurable": {"thread_id": "1"}}

response= agent.invoke(
    {"messages": [{"role":"user", "content": "What is the weather ooutside?"}]},
    config= config,
    context=Context(user_id= "1")
)

print(response['structured_response'])
# print(response)

ResponseFormat(punny_response="It's sunny in Florida, so you can really soak up the rays-itude!", weather_conditions='sunny')


In [69]:
response1 = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config,
    context=Context(user_id="1") #  same user id allows to continue the conversation
)

# print(response1['structured_response'])
print(response1['messages'][-1].content)

You're as refreshing as a rain shower in summer! Let me know if you need any more weather wonders! 🌦️


In [63]:
response1

{'messages': [HumanMessage(content='What is the weather ooutside?', additional_kwargs={}, response_metadata={}, id='0db60889-e0e1-48f2-81cd-08e673a662df'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user is asking about the weather "outside" but hasn\'t specified a location. Since they said "outside" which naturally means where they are currently located, I should get their location using the get_user_location tool first. Then I can get the weather for that location.\n\nLet me call get_user_location to find out where they are.', 'reasoning_details': [{'type': 'reasoning.text', 'text': 'The user is asking about the weather "outside" but hasn\'t specified a location. Since they said "outside" which naturally means where they are currently located, I should get their location using the get_user_location tool first. Then I can get the weather for that location.\n\nLet me call get_user_location to find out where they are.', 'format': 'unknown', 'index': 0.0}]}, res